In [ ]:
# ==========================================
# CELL 1: Install Dependencies
# ==========================================
!pip install -q transformers datasets accelerate evaluate jiwer soundfile librosa seaborn matplotlib

In [ ]:
# ==========================================
# CELL 2: Imports & Base Model Initialization
# ==========================================
import os
import re
import glob
import warnings
import dataclasses
from typing import Any, Dict, List, Union

import numpy as np
import pandas as pd
import torch
import torchaudio
import evaluate
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

# ── Suppress non-critical warnings ────────────────────────────────────────────
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ── Matplotlib Bangla-compatible font setup ───────────────────────────────────
try:
    import matplotlib.font_manager as fm
    _available_fonts = {f.name for f in fm.fontManager.ttflist}
    for _candidate in ["Noto Sans Bengali", "Noto Sans", "FreeSans", "DejaVu Sans"]:
        if _candidate in _available_fonts:
            matplotlib.rcParams["font.family"] = _candidate
            print(f"Matplotlib font set to: {_candidate}")
            break
except Exception:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "bangla-speech-processing/BanglaASR"

print(f"Using device: {device}")

processor = WhisperProcessor.from_pretrained(model_path)
model     = WhisperForConditionalGeneration.from_pretrained(model_path).to(device)

In [ ]:
# ==========================================
# CELL 3: District / Division Lookup Tables
# ==========================================
#
# The Train_annotation CSVs have columns:  audio | text | Division
# where Division is the English district/division name, e.g. "Barisal".
# Audio filenames encode the district token, e.g. "female_barisal_1.wav".
#
# We build two mappings (keyed by lowercase English district token):
#   DISTRICT_TO_BANGLA_NAME  →  Bangla name shown in outputs (জেলা)
#   DISTRICT_TO_BANGLA_DIV   →  Bangla division name          (বিভাগ)
# ─────────────────────────────────────────────────────────────────────────────

DISTRICT_TO_BANGLA_NAME: Dict[str, str] = {
    # ── Barisal Division ──────────────────────────────────────────────────────
    "barisal":           "বরিশাল",
    "bhola":             "ভোলা",
    "jhalokati":         "ঝালকাঠি",
    "patuakhali":        "পটুয়াখালী",
    "barguna":           "বরগুনা",
    "pirojpur":          "পিরোজপুর",
    # ── Chittagong Division ───────────────────────────────────────────────────
    "chittagong":        "চট্টগ্রাম",
    "comilla":           "কুমিল্লা",
    "feni":              "ফেনী",
    "brahmanbaria":      "ব্রাহ্মণবাড়িয়া",
    "noakhali":          "নোয়াখালী",
    "laksmipur":         "লক্ষ্মীপুর",
    "lakhsmipur":        "লক্ষ্মীপুর",
    "chandpur":          "চাঁদপুর",
    "coxsbazar":         "কক্সবাজার",
    # ── Dhaka Division ────────────────────────────────────────────────────────
    "dhaka":             "ঢাকা",
    "faridpur":          "ফরিদপুর",
    "gazipur":           "গাজীপুর",
    "kishoreganj":       "কিশোরগঞ্জ",
    "manikganj":         "মানিকগঞ্জ",
    "munshiganj":        "মুন্সিগঞ্জ",
    "narayanganj":       "নারায়ণগঞ্জ",
    "narsingdi":         "নরসিংদী",
    "rajbari":           "রাজবাড়ী",
    "shariatpur":        "শরীয়তপুর",
    "tangail":           "টাঙ্গাইল",
    # ── Mymensingh Division ───────────────────────────────────────────────────
    "mymensingh":        "ময়মনসিংহ",
    "netrokona":         "নেত্রকোণা",
    "jamalpur":          "জামালপুর",
    "sherpur":           "শেরপুর",
    # ── Khulna Division ───────────────────────────────────────────────────────
    "khulna":            "খুলনা",
    "jessore":           "যশোর",
    "kushtia":           "কুষ্টিয়া",
    "satkhira":          "সাতক্ষীরা",
    "bagerhat":          "বাগেরহাট",
    "chuadanga":         "চুয়াডাঙ্গা",
    "jhenaidah":         "ঝিনাইদহ",
    "magura":            "মাগুরা",
    "meherpur":          "মেহেরপুর",
    "narail":            "নড়াইল",
    # ── Rajshahi Division ─────────────────────────────────────────────────────
    "rajshahi":          "রাজশাহী",
    "bogura":            "বগুড়া",
    "bogra":             "বগুড়া",
    "chapainawabganj":   "চাঁপাইনবাবগঞ্জ",
    "joypurhat":         "জয়পুরহাট",
    "naogaon":           "নওগাঁ",
    "natore":            "নাটোর",
    "pabna":             "পাবনা",
    "sirajganj":         "সিরাজগঞ্জ",
    # ── Rangpur Division ──────────────────────────────────────────────────────
    "rangpur":           "রংপুর",
    "dinajpur":          "দিনাজপুর",
    "gaibandha":         "গাইবান্ধা",
    "kurigram":          "কুড়িগ্রাম",
    "lalmonirhat":       "লালমনিরহাট",
    "nilphamari":        "নীলফামারী",
    "panchagarh":        "পঞ্চগড়",
    "thakurgaon":        "ঠাকুরগাঁও",
    # ── Sylhet Division ───────────────────────────────────────────────────────
    "sylhet":            "সিলেট",
    "habiganj":          "হবিগঞ্জ",
    "moulvibazar":       "মৌলভীবাজার",
    "sunamganj":         "সুনামগঞ্জ",
}

DISTRICT_TO_BANGLA_DIV: Dict[str, str] = {
    "barisal": "বরিশাল",  "bhola": "বরিশাল",    "jhalokati": "বরিশাল",
    "patuakhali": "বরিশাল", "barguna": "বরিশাল", "pirojpur": "বরিশাল",
    "chittagong": "চট্টগ্রাম", "comilla": "চট্টগ্রাম", "feni": "চট্টগ্রাম",
    "brahmanbaria": "চট্টগ্রাম", "noakhali": "চট্টগ্রাম",
    "laksmipur": "চট্টগ্রাম", "lakhsmipur": "চট্টগ্রাম",
    "chandpur": "চট্টগ্রাম", "coxsbazar": "চট্টগ্রাম",
    "dhaka": "ঢাকা", "faridpur": "ঢাকা", "gazipur": "ঢাকা",
    "kishoreganj": "ঢাকা", "manikganj": "ঢাকা", "munshiganj": "ঢাকা",
    "narayanganj": "ঢাকা", "narsingdi": "ঢাকা", "rajbari": "ঢাকা",
    "shariatpur": "ঢাকা", "tangail": "ঢাকা",
    "mymensingh": "ময়মনসিংহ", "netrokona": "ময়মনসিংহ",
    "jamalpur": "ময়মনসিংহ", "sherpur": "ময়মনসিংহ",
    "khulna": "খুলনা", "jessore": "খুলনা", "kushtia": "খুলনা",
    "satkhira": "খুলনা", "bagerhat": "খুলনা", "chuadanga": "খুলনা",
    "jhenaidah": "খুলনা", "magura": "খুলনা", "meherpur": "খুলনা",
    "narail": "খুলনা",
    "rajshahi": "রাজশাহী", "bogura": "রাজশাহী", "bogra": "রাজশাহী",
    "chapainawabganj": "রাজশাহী", "joypurhat": "রাজশাহী",
    "naogaon": "রাজশাহী", "natore": "রাজশাহী",
    "pabna": "রাজশাহী", "sirajganj": "রাজশাহী",
    "rangpur": "রংপুর", "dinajpur": "রংপুর", "gaibandha": "রংপুর",
    "kurigram": "রংপুর", "lalmonirhat": "রংপুর", "nilphamari": "রংপুর",
    "panchagarh": "রংপুর", "thakurgaon": "রংপুর",
    "sylhet": "সিলেট", "habiganj": "সিলেট",
    "moulvibazar": "সিলেট", "sunamganj": "সিলেট",
}


def district_token_from_filename(filepath: str) -> str:
    """
    Extract the lowercase district token from an audio filename.

    Examples
    --------
    female_barisal_1.wav   →  'barisal'
    male_bhola_007.wav     →  'bhola'
    sylhet_female_03.wav   →  'sylhet'
    """
    stem  = os.path.splitext(os.path.basename(filepath))[0].lower()
    parts = stem.split("_")

    # Remove leading gender token (male / female / m / f)
    gender_tokens = {"male", "female", "m", "f"}
    if parts and parts[0] in gender_tokens:
        parts = parts[1:]
    # Remove trailing purely numeric token
    if parts and parts[-1].isdigit():
        parts = parts[:-1]

    # Remaining middle part(s) form the district token
    token = "_".join(parts)

    # If exact match exists, return it
    if token in DISTRICT_TO_BANGLA_NAME:
        return token

    # Fallback: check if any known key appears as a substring
    for key in DISTRICT_TO_BANGLA_NAME:
        if key in token:
            return key

    return token  # return as-is (will appear in output as raw token)


def bangla_district(token: str) -> str:
    return DISTRICT_TO_BANGLA_NAME.get(token, token)


def bangla_division(token: str) -> str:
    return DISTRICT_TO_BANGLA_DIV.get(token, "অজানা")


print("Lookup tables loaded.")
print(f"Total districts mapped: {len(DISTRICT_TO_BANGLA_NAME)}")

In [ ]:
# ==========================================
# CELL 4: Data Parsing & Custom PyTorch Dataset
#
# CSV format:  audio | text | Division
#   - 'Division' column holds the English district name, e.g. "Barisal"
#   - We also derive the district token from the audio filename
#     (e.g. female_barisal_1.wav → 'barisal') and store Bangla labels.
# ==========================================
dataset_dir    = "/kaggle/input/datasets/prosenjitmondol/bangla-regional/shobdotori"
train_dir      = os.path.join(dataset_dir, "Train")
annotation_dir = os.path.join(dataset_dir, "Train_annotation")

csv_files = glob.glob(os.path.join(annotation_dir, "*.csv"))
dfs = []

for csv_path in csv_files:
    folder_name = os.path.splitext(os.path.basename(csv_path))[0]   # e.g. "Barisal"
    audio_dir   = os.path.join(train_dir, folder_name)

    sub = pd.read_csv(csv_path)

    # Normalise column names (handle header variations / misspellings)
    sub.columns = [c.strip() for c in sub.columns]
    audio_col  = sub.columns[0]   # always first
    text_col   = sub.columns[1]   # always second
    # Third column may be 'Division', 'Divison', 'region', etc.
    div_col = sub.columns[2] if len(sub.columns) >= 3 else None

    keep = [audio_col, text_col] + ([div_col] if div_col else [])
    sub  = sub[keep].copy()

    if div_col:
        sub.columns = ["audio_id", "sentence", "division_en"]
    else:
        sub.columns = ["audio_id", "sentence"]
        sub["division_en"] = folder_name   # fallback

    # Build full audio path
    def make_path(val, base=audio_dir):
        fn = str(val).strip()
        if not fn.lower().endswith(".wav"):
            fn += ".wav"
        return os.path.join(base, fn)

    sub["audio_path"]  = sub["audio_id"].apply(make_path)

    # Derive district token from filename  (e.g. female_barisal_1 → 'barisal')
    sub["district_token"] = sub["audio_id"].apply(district_token_from_filename)

    # Bangla labels derived from the district token
    sub["জেলা"]   = sub["district_token"].apply(bangla_district)
    sub["বিভাগ"]  = sub["district_token"].apply(bangla_division)

    dfs.append(sub)

full_df = pd.concat(dfs, ignore_index=True).dropna(subset=["sentence"])
full_df = full_df[full_df["audio_path"].apply(os.path.exists)].reset_index(drop=True)

print(f"Total valid audio samples: {len(full_df)}")
print("\nSamples per district (top 20):")
counts = (
    full_df.groupby(["district_token", "জেলা", "বিভাগ"])["audio_id"]
    .count()
    .reset_index()
    .rename(columns={"audio_id": "count"})
    .sort_values("count", ascending=False)
)
print(counts.head(20).to_string(index=False))

# ── Train / Eval split (90 / 10) ─────────────────────────────────────────────
train_df, eval_df = train_test_split(full_df, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
eval_df  = eval_df.reset_index(drop=True)

print(f"\nTrain samples : {len(train_df)}")
print(f"Eval  samples : {len(eval_df)}")

# ── PyTorch Dataset ───────────────────────────────────────────────────────────
class BanglaASRDataset(Dataset):
    """Dynamic audio dataset — loads & resamples on the fly."""

    def __init__(self, df: pd.DataFrame, processor):
        self.df        = df
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        waveform, sr = torchaudio.load(row["audio_path"])

        if waveform.shape[0] > 1:                         # stereo → mono
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sr != 16000:                                    # resample to 16 kHz
            waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)

        audio_np = waveform.squeeze().numpy()

        input_features = self.processor.feature_extractor(
            audio_np, sampling_rate=16000
        ).input_features[0]

        labels = self.processor.tokenizer(str(row["sentence"])).input_ids

        return {"input_features": input_features, "labels": labels}


train_dataset = BanglaASRDataset(train_df, processor)
eval_dataset  = BanglaASRDataset(eval_df,  processor)

In [ ]:
# ==========================================
# CELL 5: Data Collator & Metric
# ==========================================
@dataclasses.dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_feats = [{"input_features": f["input_features"]} for f in features]
        batch       = self.processor.feature_extractor.pad(input_feats, return_tensors="pt")

        label_feats  = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_feats, return_tensors="pt")
        labels       = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
wer_metric    = evaluate.load("wer")


def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    return {"wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str)}

In [ ]:
# ==========================================
# CELL 6: Fine-Tuning
# ==========================================
# Place generation flags on generation_config (not model.config) to avoid
# ValueError in newer transformers versions.
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens    = []
model.config.use_cache = False

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/bangla_asr_checkpoints",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=500,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=100,
    eval_steps=100,
    logging_steps=25,
    report_to=["none"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=2,
    dataloader_num_workers=2,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)


def _find_last_checkpoint(output_dir: str):
    if not os.path.isdir(output_dir):
        return None
    ckpts = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    if not ckpts:
        return None
    return max(ckpts, key=lambda p: int(p.rsplit("-", 1)[-1]))


last_checkpoint = _find_last_checkpoint(training_args.output_dir)
if last_checkpoint:
    print(f"Resuming from checkpoint: {last_checkpoint}")
trainer.train(resume_from_checkpoint=last_checkpoint)

In [ ]:
# ==========================================
# CELL 7: Save Best Model & Validate
# ==========================================
save_path = "/kaggle/working/bangla_asr_best"
trainer.save_model(save_path)
processor.save_pretrained(save_path)
print(f"Best model saved → {save_path}")

eval_results = trainer.evaluate()
print(f"\nFinal Validation Loss : {eval_results.get('eval_loss', 0.0):.4f}")
print(f"Final Validation WER  : {eval_results.get('eval_wer',  0.0):.2f}%")

In [ ]:
# ==========================================
# CELL 8: Test Inference & Submission
#
# For every test file (e.g. female_barisal_1.wav) we:
#   1. Transcribe with the fine-tuned model
#   2. Parse the district token from the filename
#   3. Map to Bangla জেলা and বিভাগ
#   4. Save to submission.csv
# ==========================================
test_dir = "/kaggle/input/datasets/prosenjitmondol/bangla-regional/shobdotori/Test"

test_audio_files = sorted(glob.glob(os.path.join(test_dir, "**", "*.wav"), recursive=True))
if not test_audio_files:
    test_audio_files = sorted(glob.glob(os.path.join(test_dir, "*.wav")))

print(f"Found {len(test_audio_files)} test audio files.")

# Load the best model
best_model = WhisperForConditionalGeneration.from_pretrained(save_path).to(device)
best_model.eval()
best_model.generation_config.forced_decoder_ids = None
best_model.generation_config.suppress_tokens    = []

results = []
for file_path in tqdm(test_audio_files, desc="Transcribing"):
    file_id       = os.path.splitext(os.path.basename(file_path))[0]
    dist_token    = district_token_from_filename(file_path)
    bangla_jela   = bangla_district(dist_token)
    bangla_bivag  = bangla_division(dist_token)

    waveform, sr = torchaudio.load(file_path)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    if sr != 16000:
        waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)

    audio_np = waveform.squeeze().numpy()
    feats    = processor.feature_extractor(
        audio_np, sampling_rate=16000, return_tensors="pt"
    ).input_features.to(device)

    attn_mask = torch.ones(feats.shape[0], feats.shape[-1], dtype=torch.long, device=device)

    with torch.no_grad():
        pred_ids      = best_model.generate(input_features=feats)
        transcription = processor.decode(pred_ids[0], skip_special_tokens=True)

    results.append({
        "id":             file_id,
        "transcription":  transcription,
        "জেলা":           bangla_jela,
        "বিভাগ":          bangla_bivag,
    })

submission_df = pd.DataFrame(results)
submission_df.to_csv("/kaggle/working/submission.csv", index=False, encoding="utf-8-sig")

print(f"\n✓ Saved → /kaggle/working/submission.csv  ({len(submission_df)} rows)")
print("\nSample output (first 10 rows):")
print(submission_df.head(10).to_string(index=False))

In [ ]:
# ==========================================
# CELL 9: Per-District WER on Validation Set
# (full error analysis)
# ==========================================
print("Computing per-district WER on the validation split...")

wer_metric_eval = evaluate.load("wer")
best_model.eval()

val_rows = []
for idx in tqdm(range(len(eval_df)), desc="Val inference"):
    row = eval_df.iloc[idx]
    try:
        waveform, sr = torchaudio.load(row["audio_path"])
    except Exception as exc:
        print(f"[SKIP] {row['audio_path']}: {exc}")
        continue

    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    if sr != 16000:
        waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)

    audio_np = waveform.squeeze().numpy()
    feats    = processor.feature_extractor(
        audio_np, sampling_rate=16000, return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():
        pred_ids   = best_model.generate(input_features=feats)
        prediction = processor.decode(pred_ids[0], skip_special_tokens=True)

    val_rows.append({
        "district_token": row["district_token"],
        "জেলা":           row["জেলা"],
        "বিভাগ":          row["বিভাগ"],
        "reference":      str(row["sentence"]),
        "prediction":     prediction,
    })

val_detail_df = pd.DataFrame(val_rows)

# Per-district WER
wer_rows = []
for token, grp in val_detail_df.groupby("district_token"):
    refs  = grp["reference"].tolist()
    preds = grp["prediction"].tolist()
    if not refs:
        continue
    w = 100 * wer_metric_eval.compute(predictions=preds, references=refs)
    wer_rows.append({
        "District (EN)": token,
        "জেলা":          grp["জেলা"].iloc[0],
        "বিভাগ":         grp["বিভাগ"].iloc[0],
        "Samples":       len(grp),
        "WER (%)": round(w, 2),
    })

district_wer_df = (
    pd.DataFrame(wer_rows)
      .sort_values("WER (%)")
      .reset_index(drop=True)
)

print("\nPer-District WER:")
print(district_wer_df.to_string(index=False))
district_wer_df.to_csv("/kaggle/working/per_district_wer.csv",
                        index=False, encoding="utf-8-sig")
print("\n✓ Saved → /kaggle/working/per_district_wer.csv")

In [ ]:
# ==========================================
# CELL 10: Paper Figures  (all warnings fixed)
#
# Figure 1 — Training Loss & Validation WER curves
# Figure 2 — Per-district WER horizontal bar chart (Bangla labels)
# Figure 3 — Per-division WER vertical bar chart   (Bangla labels)
# Figure 4 — Styled sample-predictions table
# ==========================================
import warnings as _w
_w.filterwarnings("ignore")   # silence all remaining plt/font warnings

# ── Style & colour palette ─────────────────────────────────────────────────────
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    try:
        plt.style.use("seaborn-whitegrid")
    except OSError:
        plt.style.use("ggplot")

C = {
    "blue":   "#2563EB",
    "violet": "#7C3AED",
    "pink":   "#DB2777",
    "green":  "#059669",
    "amber":  "#D97706",
    "teal":   "#0891B2",
    "bg":     "#F8FAFC",
    "grid":   "#CBD5E1",
}

SAVE_DIR = "/kaggle/working"

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 1 : Training curves
# ─────────────────────────────────────────────────────────────────────────────
log_history = trainer.state.log_history if hasattr(trainer, "state") else []

train_steps, train_losses = [], []
eval_steps,  eval_losses  = [], []
eval_wers                 = []

for entry in log_history:
    if "loss" in entry and "eval_loss" not in entry:
        train_steps.append(entry["step"])
        train_losses.append(entry["loss"])
    if "eval_loss" in entry:
        eval_steps.append(entry["step"])
        eval_losses.append(entry["eval_loss"])
        eval_wers.append(entry.get("eval_wer", None))

if train_steps:   # only plot if we have real log data
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=C["bg"])
    for ax in axes:
        ax.set_facecolor(C["bg"])

    axes[0].plot(train_steps, train_losses,
                 color=C["blue"],  lw=2,   label="Training Loss",   zorder=3)
    axes[0].plot(eval_steps,  eval_losses,
                 color=C["pink"],  lw=2.5, marker="o", ms=5,
                 label="Validation Loss", zorder=4)
    axes[0].set_title("Training & Validation Loss", fontsize=14, fontweight="bold", pad=10)
    axes[0].set_xlabel("Step", fontsize=11)
    axes[0].set_ylabel("Loss", fontsize=11)
    axes[0].legend(fontsize=10)
    axes[0].grid(True, color=C["grid"], lw=0.8)

    wer_vals = [w for w in eval_wers if w is not None]
    wer_stps = [s for s, w in zip(eval_steps, eval_wers) if w is not None]
    axes[1].plot(wer_stps, wer_vals,
                 color=C["violet"], lw=2.5, marker="s", ms=5, zorder=4)
    axes[1].set_title("Validation WER over Steps", fontsize=14, fontweight="bold", pad=10)
    axes[1].set_xlabel("Step", fontsize=11)
    axes[1].set_ylabel("WER (%)", fontsize=11)
    axes[1].grid(True, color=C["grid"], lw=0.8)

    fig.tight_layout(pad=2.5)
    fig.savefig(f"{SAVE_DIR}/fig1_training_curves.png", dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved → fig1_training_curves.png")
else:
    print("[INFO] No training log data found — skipping Figure 1.")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 2 : Per-district WER  (horizontal bar, Bangla labels)
# ─────────────────────────────────────────────────────────────────────────────
if len(district_wer_df) > 0:
    plot_df = district_wer_df.sort_values("WER (%)", ascending=True).reset_index(drop=True)
    n       = len(plot_df)
    # Colour gradient low-WER=green → high-WER=red
    cmap   = plt.cm.get_cmap("RdYlGn_r", n)
    colors = [cmap(i / max(n - 1, 1)) for i in range(n)]

    fig, ax = plt.subplots(figsize=(11, max(5, n * 0.45)), facecolor=C["bg"])
    ax.set_facecolor(C["bg"])

    labels = plot_df["জেলা"].tolist()
    values = plot_df["WER (%)"].tolist()

    bars = ax.barh(labels, values, color=colors, edgecolor="white", height=0.68, zorder=3)

    for bar, val in zip(bars, values):
        ax.text(
            val + 0.4, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", ha="left", fontsize=8, color="#1E293B"
        )

    ax.set_xlabel("Word Error Rate (%)", fontsize=12)
    ax.set_title("Per-District WER on Validation Set",
                 fontsize=14, fontweight="bold", pad=12)
    ax.set_xlim(0, max(values) + 10)
    ax.grid(axis="x", color=C["grid"], lw=0.8, zorder=0)
    ax.invert_yaxis()

    fig.tight_layout(pad=2)
    fig.savefig(f"{SAVE_DIR}/fig2_per_district_wer.png", dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved → fig2_per_district_wer.png")
else:
    print("[INFO] No district WER data — skipping Figure 2.")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 3 : Per-division WER  (vertical bar, Bangla labels)
# ─────────────────────────────────────────────────────────────────────────────
if len(val_detail_df) > 0:
    div_rows = []
    for div, grp in val_detail_df.groupby("বিভাগ"):
        refs  = grp["reference"].tolist()
        preds = grp["prediction"].tolist()
        if not refs:
            continue
        w = 100 * wer_metric_eval.compute(predictions=preds, references=refs)
        div_rows.append({"বিভাগ": div, "Samples": len(grp), "WER (%)": round(w, 2)})

    div_df   = pd.DataFrame(div_rows).sort_values("WER (%)")
    bar_pool = [C["blue"], C["violet"], C["green"], C["pink"],
                C["amber"], C["teal"], "#BE185D", "#065F46"]
    n_div    = len(div_df)
    colors   = [bar_pool[i % len(bar_pool)] for i in range(n_div)]

    fig, ax = plt.subplots(figsize=(11, 5), facecolor=C["bg"])
    ax.set_facecolor(C["bg"])

    x    = list(range(n_div))
    bars = ax.bar(x, div_df["WER (%)"].tolist(),
                  color=colors, width=0.55, edgecolor="white", zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels(div_df["বিভাগ"].tolist(), fontsize=10)

    for bar, val in zip(bars, div_df["WER (%)"].tolist()):
        ax.text(
            bar.get_x() + bar.get_width() / 2, val + 0.5,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold"
        )

    ax.set_ylabel("Word Error Rate (%)", fontsize=12)
    ax.set_title("Per-Division WER on Validation Set",
                 fontsize=14, fontweight="bold", pad=12)
    ax.set_ylim(0, div_df["WER (%)"].max() + 10)
    ax.grid(axis="y", color=C["grid"], lw=0.8, zorder=0)

    fig.tight_layout(pad=2)
    fig.savefig(f"{SAVE_DIR}/fig3_per_division_wer.png", dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved → fig3_per_division_wer.png")
else:
    print("[INFO] No validation detail data — skipping Figure 3.")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 4 : Sample predictions table  (10 rows)
# ─────────────────────────────────────────────────────────────────────────────
if len(val_detail_df) >= 1:
    sample = val_detail_df[["জেলা", "বিভাগ", "reference", "prediction"]].head(10).copy()
    sample.columns = ["জেলা", "বিভাগ", "Reference (Ground Truth)", "Model Prediction"]

    fig, ax = plt.subplots(figsize=(18, 5), facecolor=C["bg"])
    ax.axis("off")

    tbl = ax.table(
        cellText=sample.values,
        colLabels=sample.columns.tolist(),
        loc="center",
        cellLoc="left",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1.0, 2.4)

    # Column widths  (জেলা, বিভাগ, Reference, Prediction)
    col_widths_map = {0: 0.09, 1: 0.09, 2: 0.41, 3: 0.41}
    for (row_i, col_i), cell in tbl.get_celld().items():
        cell.set_linewidth(0.5)
        cell.set_edgecolor(C["grid"])
        if col_i in col_widths_map:
            cell.set_width(col_widths_map[col_i])

    # Header style
    for j in range(len(sample.columns)):
        tbl[0, j].set_facecolor(C["blue"])
        tbl[0, j].set_text_props(color="white", fontweight="bold")

    # Alternating row colour
    for i in range(1, len(sample) + 1):
        row_color = "#EFF6FF" if i % 2 == 0 else "white"
        for j in range(len(sample.columns)):
            tbl[i, j].set_facecolor(row_color)

    ax.set_title(
        "Sample Model Predictions with জেলা (District) & বিভাগ (Division)",
        fontsize=13, fontweight="bold", pad=14, y=0.98
    )

    fig.tight_layout()
    fig.savefig(f"{SAVE_DIR}/fig4_sample_predictions.png", dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved → fig4_sample_predictions.png")
else:
    print("[INFO] No validation detail data — skipping Figure 4.")

print("\n✓ All paper figures saved to /kaggle/working/")